In [30]:
import time
import mysql.connector
from collections import defaultdict
from datetime import datetime

# --- Configuration ---
DB_CONFIG = {
    "database": "gossipdb",
    "user": "root",
    "password": "password",
    "host": "localhost",
    "port": 3306  # Default MySQL port
}

In [ ]:
import time
import mysql.connector
from datetime import datetime
from collections import defaultdict

# --- CONFIGURATION ---
POLL_INTERVAL = 5 # seconds

def get_db_connection():
    return mysql.connector.connect(**DB_CONFIG)

def format_ts(ts):
    """Safely formats the timestamp into a readable string."""
    # If the timestamp is a huge number, it's likely in milliseconds instead of seconds
    # Divide by 1000 so fromtimestamp() doesn't throw an OutOfRange error!
    if ts > 1e11:  
        ts = ts / 1000.0
    try:
        return datetime.fromtimestamp(ts).strftime('%H:%M:%S.%f')[:-3]
    except Exception as e:
        return str(ts)

def get_active_nodes(cursor):
    """Finds all currently active nodes based on their last START action."""
    query = """
        WITH RankedActions AS (
            SELECT 
                CONCAT(host, ':', actionPort) AS node_address, 
                action,
                ROW_NUMBER() OVER(PARTITION BY host, actionPort ORDER BY timestamp DESC) as rn
            FROM ActionRecord
        )
        SELECT node_address 
        FROM RankedActions 
        WHERE rn = 1 AND action = 'START';
    """
    cursor.execute(query)
    return {row[0] for row in cursor.fetchall()}

def get_gossip_timestamps(cursor):
    """
    Extracts the EXACT timestamp each node first saw a specific gossip UID.
    Returns: { uid: { node_address: {"forwarderTimestamp": ts, "ttl": max_ttl, "creationTimestamp": creation_ts} } }
    """
    query = """
        SELECT uid, node_address, MIN(forwarderTimestamp) as min_ts, MAX(ttl) as max_ttl, MIN(creationTimestamp) as creation_ts
        FROM (
            SELECT uid, creatorAddress AS node_address, forwarderTimestamp, ttl, creationTimestamp FROM GossipRecord WHERE creatorAddress IS NOT NULL
            UNION ALL
            SELECT uid, forwarderAddress AS node_address, forwarderTimestamp, ttl, creationTimestamp FROM GossipRecord WHERE forwarderAddress IS NOT NULL
        ) AS combined
        GROUP BY uid, creationTimestamp, node_address;
    """
    cursor.execute(query)
    
    gossip_data = defaultdict(dict)
    for row in cursor.fetchall():
        uid, node_address, raw_ts, max_ttl, creation_ts = row
        
        # Handle both MySQL DATETIME objects and raw numbers (Unix epoch)
        if isinstance(raw_ts, datetime):
            ts_val = raw_ts.timestamp() 
        else:
            ts_val = float(raw_ts)
            
        gossip_data[uid][node_address] = {
            "forwarderTimestamp": ts_val, 
            "ttl": max_ttl, 
            "creationTimestamp": creation_ts
        }
        
    return gossip_data

def monitor_convergence():
    print("🔍 Starting diagnostic convergence monitor...")
    
    reported_uids = set()
    stuck_uids = set()
    expired_uids = set()
    
    while True:
        try:
            with get_db_connection() as conn:
                with conn.cursor() as cursor:
                    
                    active_nodes = get_active_nodes(cursor)
                    total_active = len(active_nodes)
                    
                    if total_active == 0:
                        print(f"[{time.strftime('%X')}] ⚠️ No active nodes found.")
                        time.sleep(POLL_INTERVAL)
                        continue
                        
                    gossip_data = get_gossip_timestamps(cursor)
                    
                    for uid, node_data in gossip_data.items():
                        if not node_data:
                            continue
                            
                        # 1. Safely extract TTL and Creation TS from the first available node's record
                        first_node_record = next(iter(node_data.values()))
                        ttl_seconds = float(first_node_record["ttl"]) / 1_000_000_000.0
                        
                        creation_ts_raw = first_node_record["creationTimestamp"]
                        if isinstance(creation_ts_raw, datetime):
                            creation_ts_sec = creation_ts_raw.timestamp()
                        else:
                            creation_ts_sec = float(creation_ts_raw)
                            
                        # 2. Check if this UID has expired based on math
                        is_expired = (creation_ts_sec + ttl_seconds) < time.time()
                        
                        # Filter to only the nodes currently marked ACTIVE in the database
                        active_nodes_with_gossip = {node: ts_data for node, ts_data in node_data.items() if node in active_nodes}
                        total_converged = len(active_nodes_with_gossip)
                        
                        # --- SUCCESS SCENARIO ---
                        if total_converged == total_active:
                            if uid in stuck_uids:
                                print(f"✅ [UNSTUCK] UID {uid} has now converged at all nodes.")
                                stuck_uids.remove(uid)

                            if uid not in reported_uids:
                                first_seen = min([data["forwarderTimestamp"] for data in node_data.values()])
                                last_seen = max([data["forwarderTimestamp"] for data in node_data.values()])
                                duration = last_seen - first_seen
                                unit = "ms" if first_seen > 1e11 else "seconds"
                                
                                print(f"🎉 [CONVERGED] UID: {uid}")
                                print(f"   ┣ First Seen:         {format_ts(first_seen)}")
                                print(f"   ┣ 100% Converged At:  {format_ts(last_seen)}")
                                print(f"   ┗ Time to Converge:   {duration:.2f} {unit}\n")
                                reported_uids.add(uid)
                                
                        # --- FAILED / IN PROGRESS SCENARIO ---
                        else:
                            # If it's mathematically dead, report expiration
                            if is_expired:
                                if uid not in expired_uids:
                                    print(f"⏰ [EXPIRED] UID {uid} has expired without converging (Stuck at {total_converged}/{total_active}).")
                                    expired_uids.add(uid)
                                    # Cleanup from stuck tracking if it died while stuck
                                    if uid in stuck_uids:
                                        stuck_uids.remove(uid)
                            # If it's still alive, report it as stuck/propagating
                            else:
                                if uid not in stuck_uids and uid not in expired_uids:
                                    missing_nodes = active_nodes - set(active_nodes_with_gossip.keys())
                                    print(f"⚠️ [STUCK] UID {uid} is at {total_converged}/{total_active} nodes.")
                                    print(f"   ┗ Missing on: {', '.join(missing_nodes)}")
                                    stuck_uids.add(uid)
                
        except mysql.connector.Error as err:
            print(f"\n❌ MySQL error: {err}")
        except Exception as e:
            print(f"\n❌ Unexpected error: {e}")
            
        time.sleep(POLL_INTERVAL)

if __name__ == "__main__":
    monitor_convergence()

🔍 Starting diagnostic convergence monitor...
[15:07:54] ⚠️ No active nodes found.
[15:07:59] ⚠️ No active nodes found.
⚠️ [STUCK] UID 04192f1a-d325-06e8-2118-b95b1fdc3271 is at 3/4 nodes.
   ┗ Missing on: localhost:9002
⚠️ [STUCK] UID 9d225e61-f719-6c44-89eb-2c54a942f01f is at 3/4 nodes.
   ┗ Missing on: localhost:9000
🎉 [CONVERGED] UID: 9a454bcf-4e6a-1bc8-f97f-1710eb74b51d
   ┣ First Seen:         18:07:44.331
   ┣ 100% Converged At:  18:07:44.683
   ┗ Time to Converge:   0.35 seconds

🎉 [CONVERGED] UID: 5583f288-fbda-d621-cac1-3d1694cdad53
   ┣ First Seen:         18:07:44.508
   ┣ 100% Converged At:  18:07:44.683
   ┗ Time to Converge:   0.18 seconds

⚠️ [STUCK] UID 44806dfa-70fe-a90b-d78a-b356da987e3d is at 3/4 nodes.
   ┗ Missing on: localhost:9002
🎉 [CONVERGED] UID: 0cb82f42-f0f0-128c-5394-0febcc2b640e
   ┣ First Seen:         18:07:44.683
   ┣ 100% Converged At:  18:07:44.857
   ┗ Time to Converge:   0.17 seconds

🎉 [CONVERGED] UID: 495f116a-db40-f039-1966-0df55b9bcac2
   ┣ Firs

KeyboardInterrupt: 